# Serialization

Every object the pipeline takes in or hands back carries the same pair of methods.
`dump_json(indent)` returns a JSON string, and the class method `from_json` rebuilds an object
from one.

In [1]:
import json
import math

import sympy as sp

from lin_ode.basis import CarlemanBasis, GaussianHermiteBasis
from lin_ode.matrix import Matrix, OperatorMatrix
from lin_ode.observable import ObservableExpression
from lin_ode.operators import Koopman, LinearOperator
from lin_ode.ordinary_differential_system import (
    DerivativeCondition,
    DirichletCondition,
    Facet,
    OrdinaryDifferentialSystem,
)
from lin_ode.vector import Vector

u, i = sp.IndexedBase("u"), sp.Idx("i", (0, 3))

## The ODE

In [2]:
ode = OrdinaryDifferentialSystem.from_sympy(
    u[i] ** 2,
    u,
    (i,),
    conditions=[
        DirichletCondition(Facet.low(i), sp.Rational(1, 2)),
        DerivativeCondition(Facet.high(i), sp.Rational(1, 3)),
    ],
)
json_str = ode.dump_json(indent=2)
print(json_str)

reloaded = OrdinaryDifferentialSystem.from_json(json_str)
print("\nreloaded equation:", reloaded.equation.expr)

{
  "equation": {
    "expr": "Piecewise(ExprCondPair(Integer(0), Equality(Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3))), Integer(0))), ExprCondPair(Rational(1, 3), Equality(Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3))), Integer(3))), ExprCondPair(Pow(Indexed(IndexedBase(Symbol('u')), Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3)))), Integer(2)), true))",
    "state_variable": "IndexedBase(Symbol('u'))",
    "indices": "(Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3))),)"
  },
  "conditions": [
    {
      "type": "DirichletCondition",
      "region": {
        "type": "Facet",
        "index": "Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3)))",
        "position": 0
      },
      "value": "Rational(1, 2)"
    },
    {
      "type": "DerivativeCondition",
      "region": {
        "type": "Facet",
        "index": "Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3)))",
        "position": 3
      },
     

## The linearized operator

A linearized operator comes back through `LinearOperator.from_json`.

In [3]:
operator = Koopman().create_operator(ode)
json_str = operator.dump_json(indent=2)
print(json_str)

reloaded = LinearOperator.from_json(json_str)
print("\nreloaded method:", reloaded.method_name)
print("reloaded arms  :", reloaded.cases)

{
  "type": "OperatorSum",
  "operator": {
    "expr": "Piecewise(ExprCondPair(Integer(0), Equality(Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3))), Integer(0))), ExprCondPair(Mul(Rational(1, 3), Indexed(IndexedBase(Symbol('K', commutative=False)), Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3))))), Equality(Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3))), Integer(3))), ExprCondPair(Mul(Pow(Indexed(IndexedBase(Symbol('U', commutative=False)), Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3)))), Integer(2)), Indexed(IndexedBase(Symbol('K', commutative=False)), Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3))))), true))",
    "state_variable": "IndexedBase(Symbol('U', commutative=False))",
    "indices": "(Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3))),)"
  },
  "diff_base": "IndexedBase(Symbol('K', commutative=False))",
  "method_name": "Koopman",
  "requires_l2_basis": false
}

reloaded method: Koopman
reloa

## The basis

Each basis class restores its own through `from_json`. The string records the shape arguments,
and the pivot for PSC, but not the projectors: a basis built with `projectors=` comes back
with the ones its class ships ([`06_vector`](06_vector.ipynb) builds such a basis).

In [4]:
basis = CarlemanBasis(4, 2)
json_str = basis.dump_json(indent=2)
print(json_str)

reloaded = CarlemanBasis.from_json(json_str)
print("\nstates in the reloaded basis:", len(reloaded))

{
  "type": "CarlemanBasis",
  "n_vars": 4,
  "n_vars_is_tuple": false,
  "degree": 2
}

states in the reloaded basis: 15


## The coefficient vector

An initial distribution and an objective function both take this form.

In [5]:
vector = Vector(u[0] ** 2, basis, u)
json_str = vector.dump_json(indent=2)
print(json_str)

reloaded = Vector.from_json(json_str)
print("\ncoefficient of u_0**2:", reloaded[(0, 2)])

{
  "vector": "Pow(Indexed(IndexedBase(Symbol('u')), Integer(0)), Integer(2))",
  "basis": {
    "type": "CarlemanBasis",
    "n_vars": 4,
    "n_vars_is_tuple": false,
    "degree": 2
  },
  "u": "IndexedBase(Symbol('u'))"
}

coefficient of u_0**2: 1


A fixed initial value is a list of numbers instead, so `json` handles it
with no help from `lin_ode`.

In [6]:
x0 = [0.3, 0.7, 1.1, 0.5]
y0_entries = [
    math.prod(x0[int(site)] ** int(power) for site, power in basis[label].args)
    for label in [(), ((0, 1),), ((0, 2),)]
]
json_str = json.dumps(y0_entries, indent=2)
print(json_str)

print("\nreloaded entries:", json.loads(json_str))

[
  1,
  0.3,
  0.09
]

reloaded entries: [1, 0.3, 0.09]


## The operator matrix

In [7]:
matrix = OperatorMatrix(basis, operator)
json_str = matrix.dump_json(indent=2)
print(json_str)

reloaded = OperatorMatrix.from_json(json_str)
print("\nelement of the original:", matrix[((1, 2),), ((1, 1),)])
print("element after reloading:", reloaded[((1, 2),), ((1, 1),)])

{
  "basis": {
    "type": "CarlemanBasis",
    "n_vars": 4,
    "n_vars_is_tuple": false,
    "degree": 2
  },
  "operator": {
    "type": "OperatorSum",
    "operator": {
      "expr": "Piecewise(ExprCondPair(Integer(0), Equality(Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3))), Integer(0))), ExprCondPair(Mul(Rational(1, 3), Indexed(IndexedBase(Symbol('K', commutative=False)), Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3))))), Equality(Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3))), Integer(3))), ExprCondPair(Mul(Pow(Indexed(IndexedBase(Symbol('U', commutative=False)), Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3)))), Integer(2)), Indexed(IndexedBase(Symbol('K', commutative=False)), Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3))))), true))",
      "state_variable": "IndexedBase(Symbol('U', commutative=False))",
      "indices": "(Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3))),)"
    },
    "diff_base

Under KvN an observable is an operator too, and `get_matrix` returns the same `OperatorMatrix`.

In [8]:
observable = ObservableExpression.from_sympy(u[0] ** 2, u, (i,))
kvn_matrix = observable.get_matrix(GaussianHermiteBasis(4, 2))
json_str = kvn_matrix.dump_json(indent=2)
print(json_str)

reloaded = OperatorMatrix.from_json(json_str)
print("\nelement after reloading:", reloaded[(), ((0, 2),)])

{
  "basis": {
    "type": "GaussianHermiteBasis",
    "n_vars": 4,
    "n_vars_is_tuple": false,
    "degree": 2
  },
  "operator": {
    "type": "SimpleOperator",
    "operator": {
      "expr": "Pow(Indexed(IndexedBase(Symbol('u')), Integer(0)), Integer(2))",
      "state_variable": "IndexedBase(Symbol('u'))",
      "indices": "(Idx(Symbol('i', integer=True), Tuple(Integer(0), Integer(3))),)"
    },
    "diff_base": "IndexedBase(Symbol('K', commutative=False))",
    "method_name": "",
    "requires_l2_basis": false
  }
}

element after reloading: sqrt(2)/2


## The block

`matrix[rows, cols]` returns a `Matrix`, a block whose entries are already computed. Its JSON is
those entries, one `srepr` string each, so reloading the block costs no assembly.

In [9]:
labels = [(), ((3, 1),), ((3, 2),)]
block = matrix[labels, labels]
json_str = block.dump_json(indent=2)
print(json_str)

print("\nround-trips:", Matrix.from_json(json_str) == block)

[
  [
    "Integer(0)",
    "Rational(1, 3)",
    "Integer(0)"
  ],
  [
    "Integer(0)",
    "Integer(0)",
    "Rational(2, 3)"
  ],
  [
    "Integer(0)",
    "Integer(0)",
    "Integer(0)"
  ]
]

round-trips: True


---

[← End to end](08_end_to_end.ipynb) · [Notebook index](../README.md)